In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType, DateType, TimestampType

# Read bronze transactions
df_bronze = spark.table("digital_banking.bronze.bronze_transactions")

# Read valid accounts for reference validation
df_valid_accounts = spark.table("digital_banking.silver.silver_accounts").select("account_id")

print(f"Total bronze records: {df_bronze.count()}")

In [0]:
# Transform data types and add validation columns
df_transformed = df_bronze.select(
    F.col("transaction_id"),
    F.col("account_id"),
    # Convert date and timestamp strings to proper types
    F.to_date(F.col("transaction_date"), "yyyy-MM-dd").alias("transaction_date"),
    F.to_timestamp(F.col("transaction_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("transaction_timestamp"),
    F.col("transaction_type"),
    F.col("transaction_channel"),
    # Convert amount from string to decimal
    F.col("amount").cast(DecimalType(18, 2)).alias("amount"),
    F.col("currency"),
    F.col("merchant_name"),
    F.col("merchant_category"),
    F.col("transaction_status"),
    F.col("reference_number"),
    F.to_timestamp(F.col("created_at"), "yyyy-MM-dd HH:mm:ss").alias("created_at")
)

display(df_transformed.limit(5))

In [0]:
# Add row number for duplicate detection (keep first occurrence)
window_spec = Window.partitionBy("transaction_id").orderBy("transaction_timestamp")

df_with_validations = df_transformed.withColumn(
    "row_num", F.row_number().over(window_spec)
)

# Join with valid accounts to flag invalid account references
df_with_validations = df_with_validations.join(
    df_valid_accounts,
    df_with_validations.account_id == df_valid_accounts.account_id,
    "left"
).select(
    df_with_validations["*"],
    df_valid_accounts.account_id.alias("valid_account_flag")
)

# Create validation flags and rejection reasons
# For KPI accuracy: Only "Completed" transactions go to silver
# Failed and Reversed are moved to rejected table
df_validated = df_with_validations.withColumn(
    "rejection_reason",
    F.when(F.col("row_num") > 1, "Duplicate transaction_id")
    .when(F.col("valid_account_flag").isNull(), "Invalid account reference")
    .when(F.col("amount").isNull(), "Invalid amount - cannot convert to decimal")
    .when(F.col("amount") <= 0, "Invalid amount - zero or negative")
    .when(F.col("transaction_date").isNull(), "Invalid transaction_date")
    .when(F.col("transaction_timestamp").isNull(), "Invalid transaction_timestamp")
    .when(F.col("created_at").isNull(), "Invalid created_at timestamp")
    .when(F.col("transaction_status") == "Failed", "Failed transaction - excluded from silver")
    .when(F.col("transaction_status") == "Reversed", "Reversed transaction - excluded from silver")
    .when(
        ~F.col("transaction_status").isin(["Completed", "Failed", "Reversed"]),
        "Invalid transaction_status"
    )
    .otherwise(None)
).withColumn(
    "is_valid",
    F.when(F.col("rejection_reason").isNull(), True).otherwise(False)
)

# Drop helper columns
df_validated = df_validated.drop("row_num", "valid_account_flag")

# Count valid and rejected records
valid_count = df_validated.filter(F.col("is_valid") == True).count()
rejected_count = df_validated.filter(F.col("is_valid") == False).count()

print(f"Valid records: {valid_count}")
print(f"Rejected records: {rejected_count}")
print(f"\nRejection breakdown:")
display(
    df_validated.filter(F.col("is_valid") == False)
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.col("count").desc())
)

In [0]:
# Split into valid (silver) and rejected records
df_silver = df_validated.filter(F.col("is_valid") == True).select(
    "transaction_id",
    "account_id",
    "transaction_date",
    "transaction_timestamp",
    "transaction_type",
    "transaction_channel",
    "amount",
    "currency",
    "merchant_name",
    "merchant_category",
    "transaction_status",
    "reference_number",
    "created_at",
    # Add metadata columns
    F.current_timestamp().alias("processed_at"),
    F.lit("silver").alias("data_layer")
)

df_rejected = df_validated.filter(F.col("is_valid") == False).select(
    "transaction_id",
    "account_id",
    "transaction_date",
    "transaction_timestamp",
    "transaction_type",
    "transaction_channel",
    "amount",
    "currency",
    "merchant_name",
    "merchant_category",
    "transaction_status",
    "reference_number",
    "created_at",
    "rejection_reason",
    # Add metadata columns
    F.current_timestamp().alias("processed_at"),
    F.lit("rejected").alias("data_layer")
)

print(f"Silver DataFrame ready with {df_silver.count()} records")
print(f"Rejected DataFrame ready with {df_rejected.count()} records")

In [0]:
# Write valid transactions to silver table
df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.silver.silver_transactions")

print("✓ Successfully created digital_banking.silver.silver_transactions")

# Write rejected transactions to rejected table
df_rejected.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.silver.silver_transactions_rejected")